In [ ]:
# SciPy 허용 버전: n(표본수)에 따라 1집단 분산 χ² 검정 결과가 어떻게 달라지는지 비교하는 로직
# - (n-1)s^2 / sigma0^2 ~ χ²(df=n-1)  (정규 i.i.d. 표본 가정)
# - 각 n에 대해 여러 번(reps) 반복 샘플링하여
#   1) 기각률(=유의수준 또는 파워)
#   2) p-value 분포
#   3) 분산 신뢰구간(CI) 폭 및 포함률(coverage)
# 을 요약합니다.

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import chi2


def chi2_var_test(sample, sigma0_sq, alpha=0.05):
    """
    1집단 분산 χ² 검정 + 분산 CI
    H0: sigma^2 = sigma0_sq  (양측)
    반환: dict(stat, p, reject, s2, ci_lo, ci_hi, ci_width)
    """
    n = len(sample)
    df = n - 1
    s2 = np.var(sample, ddof=1)

    stat = df * s2 / sigma0_sq  # χ² 통계량
    # 양측 p-value: 2 * min(좌측누적, 우측꼬리)
    p_left = chi2.cdf(stat, df)
    p_right = chi2.sf(stat, df)  # 1 - cdf
    p_two = min(1.0, 2.0 * min(p_left, p_right))

    # (1-alpha) CI for sigma^2 using χ² quantiles
    q_lo = chi2.ppf(alpha / 2, df)
    q_hi = chi2.ppf(1 - alpha / 2, df)
    ci_lo = df * s2 / q_hi
    ci_hi = df * s2 / q_lo

    reject = p_two <= alpha

    return {
        "n": n,
        "df": df,
        "s2": s2,
        "stat": stat,
        "p": p_two,
        "reject": reject,
        "ci_lo": ci_lo,
        "ci_hi": ci_hi,
        "ci_width": (ci_hi - ci_lo),
    }


def simulate_by_n(
    n_list,
    sigma_true_sq=1.0,  # 실제(모집단) 분산
    sigma0_sq=1.0,  # 검정의 귀무가설 분산값
    alpha=0.05,
    reps=2000,
    seed=44,
):
    """
    각 n에 대해 reps번 i.i.d. 정규표본을 뽑아 χ² 분산 검정 수행 후 요약 통계 반환.
    - sigma_true_sq == sigma0_sq : 기각률이 ~alpha (제1종 오류)
    - sigma_true_sq != sigma0_sq : 기각률이 power (검정력)
    """
    rng = np.random.default_rng(seed)
    rows = []

    for n in n_list:
        ps = np.empty(reps, dtype=float)
        rejects = np.empty(reps, dtype=bool)
        ci_widths = np.empty(reps, dtype=float)
        ci_contains_true = np.empty(reps, dtype=bool)

        # i.i.d. 정규 표본 생성: N(0, sigma_true_sq)
        # 표준편차는 sqrt(분산)
        sd_true = np.sqrt(sigma_true_sq)

        for r in range(reps):
            x = rng.normal(loc=0.0, scale=sd_true, size=n)
            out = chi2_var_test(x, sigma0_sq=sigma0_sq, alpha=alpha)

            ps[r] = out["p"]
            rejects[r] = out["reject"]
            ci_widths[r] = out["ci_width"]
            ci_contains_true[r] = out["ci_lo"] <= sigma_true_sq <= out["ci_hi"]

        rows.append(
            {
                "n": n,
                "reps": reps,
                "alpha": alpha,
                "sigma_true_sq": sigma_true_sq,
                "sigma0_sq": sigma0_sq,
                "reject_rate": rejects.mean(),
                "p_median": np.median(ps),
                "p_q10": np.quantile(ps, 0.10),
                "p_q90": np.quantile(ps, 0.90),
                "ci_width_mean": ci_widths.mean(),
                "ci_width_median": np.median(ci_widths),
                "ci_coverage_true": ci_contains_true.mean(),  # CI가 true sigma^2를 포함하는 비율
            }
        )

    return pd.DataFrame(rows)


def plot_summary(df, title_prefix=""):
    # n에 따른 기각률과 CI 폭, (선택) 포함률을 시각화
    fig = plt.figure()
    plt.plot(df["n"], df["reject_rate"], marker="o")
    plt.axhline(df["alpha"].iloc[0], linestyle="--")
    plt.xlabel("n")
    plt.ylabel("Reject rate")
    plt.title(f"{title_prefix}Reject rate vs n")
    plt.show()

    fig = plt.figure()
    plt.plot(df["n"], df["ci_width_median"], marker="o")
    plt.xlabel("n")
    plt.ylabel("Median CI width for sigma^2")
    plt.title(f"{title_prefix}CI width vs n")
    plt.show()

    fig = plt.figure()
    plt.plot(df["n"], df["ci_coverage_true"], marker="o")
    plt.axhline(1 - df["alpha"].iloc[0], linestyle="--")
    plt.xlabel("n")
    plt.ylabel("Coverage (CI contains true sigma^2)")
    plt.title(f"{title_prefix}CI coverage vs n")
    plt.show()


# -----------------------------
# 1) 예시: H0가 참일 때(제1종 오류가 alpha 근처로 가는지)
# -----------------------------
n_list = [10, 20, 30, 50, 80, 120, 200, 500, 1000, 5000]
df_type1 = simulate_by_n(
    n_list=n_list,
    sigma_true_sq=1.0,  # 실제 분산
    sigma0_sq=1.0,  # H0 분산
    alpha=0.05,
    reps=10,
    seed=44,
)
print(df_type1)
plot_summary(df_type1, title_prefix="[Type I] ")

# -----------------------------
# 2) 예시: H0가 거짓일 때(파워가 n에 따라 커지는지)
#    실제 분산을 조금 키워서(예: 1.2) '분산 증가'를 잡아내는 상황
# -----------------------------
df_power = simulate_by_n(
    n_list=n_list,
    sigma_true_sq=1.2,  # 실제 분산(증가)
    sigma0_sq=1.0,  # H0 분산
    alpha=0.05,
    reps=5,
    seed=45,
)
print(df_power)
plot_summary(df_power, title_prefix="[Power] ")